In [1]:
# ============================================
# IPL Win Probability Predictor
# Day 1 — Setup & Data Loading
# Phase 2 — Environment Verification
# ============================================
# GOAL: Verify libraries, load matches.csv and 
# deliveries.csv, and look at what's inside them

In [2]:
!pip install xgboost
!pip install plotly

In [3]:
# Testing all imports
import os
import pandas as pd
import numpy as np
print("os,pandas,numpy imported successfully")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print("Scikit-learn modules imported successfully")

import xgboost as xgb
print("xgboost imported successfully")

import matplotlib.pyplot as plt
import seaborn as sns
print("matplotlib , Seasborn imported successfully")

import plotly.graph_objects as go
print("Plotly imported successfully")

print("\n All Libraries imported successfully")

os,pandas,numpy imported successfully
Scikit-learn modules imported successfully
xgboost imported successfully
matplotlib , Seasborn imported successfully
Plotly imported successfully

 All Libraries imported successfully


In [4]:
# Loading both csv files into pandas dataframe
matches=pd.read_csv("../data/matches.csv")
deliveries=pd.read_csv("../data/deliveries.csv")

print(" Matches.csv Loaded- Shape",matches.shape)
print(" Deliveries.csv Loaded- Shape",deliveries.shape)

 Matches.csv Loaded- Shape (756, 18)
 Deliveries.csv Loaded- Shape (179078, 21)


In [5]:
matches.head()

,id,Season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,IPL-2017,Hyderabad,05-04-2017,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,IPL-2017,Pune,06-04-2017,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,IPL-2017,Rajkot,07-04-2017,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,IPL-2017,Indore,08-04-2017,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,IPL-2017,Bangalore,08-04-2017,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


In [6]:
deliveries.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN


In [7]:
print("Columns in matches.csv:")
print(matches.columns.tolist())

print("Columns in deliveries.csv:")
print(deliveries.columns.tolist())

Columns in matches.csv:
['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']
Columns in deliveries.csv:
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


In [8]:
print("MATCHES DATASET INFO :")
matches.info()

print("DELIVERIES DATASET INFO :")
deliveries.info()

MATCHES DATASET INFO :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               756 non-null    int64 
 1   Season           756 non-null    object
 2   city             749 non-null    object
 3   date             756 non-null    object
 4   team1            756 non-null    object
 5   team2            756 non-null    object
 6   toss_winner      756 non-null    object
 7   toss_decision    756 non-null    object
 8   result           756 non-null    object
 9   dl_applied       756 non-null    int64 
 10  winner           752 non-null    object
 11  win_by_runs      756 non-null    int64 
 12  win_by_wickets   756 non-null    int64 
 13  player_of_match  752 non-null    object
 14  venue            756 non-null    object
 15  umpire1          754 non-null    object
 16  umpire2          754 non-null    object
 17  umpire3     

In [9]:
# Check unique seasons available
print("Seasons in data:", sorted(matches['Season'].unique()))

# Check how many matches had DL applied (rain-affected)
print("\nDL applied matches:", matches['dl_applied'].sum())
print("Total matches:", len(matches))

# Check for missing winners (e.g., abandoned matches)
print("\nMatches with no winner (abandoned/tied):", matches['winner'].isna().sum())

# Check unique result types
print("\nResult types:", matches['result'].unique())

Seasons in data: ['IPL-2008', 'IPL-2009', 'IPL-2010', 'IPL-2011', 'IPL-2012', 'IPL-2013', 'IPL-2014', 'IPL-2015', 'IPL-2016', 'IPL-2017', 'IPL-2018', 'IPL-2019']

DL applied matches: 19
Total matches: 756

Matches with no winner (abandoned/tied): 4

Result types: ['normal' 'tie' 'no result']
